# Tutorial: Supermix Colab Current Training

Audience:
- You, when you want the current Qwen Supermix training path on Google Colab without switching architectures or rewriting the trainer.

Goals:
- Reuse the same `source/qwen_supermix_pipeline.py` pipeline that the repo uses now.
- Keep Drive-backed checkpoints, prepared-data cache, teacher-distill cache, and logs alive across Colab restarts.
- Surface the newer trainer knobs that the older `supermix-colab-full-training.ipynb` notebook does not expose.

Important:
- This notebook stays on the Qwen LoRA pipeline.
- It does not switch to the legacy classifier path in `finetune_chat.py` or the retrieval/database path in `llm_database.py`.
- The optional SQLite manifest cell is only for dataset auditing in Colab. It does not change the training architecture.


## Outline

1. Verify the Colab GPU runtime.
2. Mount Drive and define persistent paths.
3. Clone the current repo snapshot with sparse checkout.
4. Install the training dependencies and cache the base model in Drive.
5. Optionally build a Drive-backed SQLite manifest for dataset auditing.
6. Build a current training command that keeps the same model architecture but exposes newer pipeline features.
   The notebook auto-tunes a few throughput knobs for common Colab GPUs like T4, L4, and A100.
7. Launch training and stream logs.
8. Reattach after reconnects and inspect checkpoints, caches, and benchmark outputs.


In [ ]:
from __future__ import annotations

import platform
import shutil
import subprocess
import sys

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
if shutil.which('nvidia-smi') is None:
    raise RuntimeError(
        'Colab is not attached to a GPU runtime. In Colab, open Runtime > Change runtime type > Hardware accelerator > GPU, then rerun from the top.'
    )
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
import os
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/supermix_colab_current')
HF_HOME = DRIVE_ROOT / 'hf_cache'
LOG_DIR = DRIVE_ROOT / 'logs'
ARTIFACT_ROOT = DRIVE_ROOT / 'artifacts'
OUTPUT_DIR = ARTIFACT_ROOT / 'qwen_supermix_enhanced_v28_colab_current'
RESUME_WARM_START_DIR = ARTIFACT_ROOT / 'qwen_supermix_enhanced_v26_full'
STATE_PATH = DRIVE_ROOT / 'last_training_launch_current.json'
SQLITE_MANIFEST_PATH = DRIVE_ROOT / 'dataset_manifest.sqlite3'
WORKSPACE_DIR = Path('/content/Supermix_27')
REPO_URL = 'https://github.com/kai9987kai/Supermix_27.git'
BRANCH = 'main'

HF_BASE_MODEL_REPO = 'Qwen/Qwen2.5-0.5B-Instruct'
HF_BASE_MODEL_REVISION = '7ae557604adf67be50417f59c2c2f167def9a775'
BASE_MODEL = str(DRIVE_ROOT / 'base_models' / f'qwen2_5_0_5b_instruct_{HF_BASE_MODEL_REVISION}')

DEVICE = 'cuda'
SAVE_EVERY_STEPS = 40
TRAIN_PROFILE = 'current_colab'  # current_colab or parity_v28
GPU_PROFILE = 'auto'  # auto, t4, l4, a100, generic
RUN_BENCHMARK_AFTER_TRAIN = False
BENCHMARK_EVAL_LIMIT = 256
ENABLE_SQLITE_DATASET_MANIFEST = True
ALLOW_BUNDLED_WARM_START_FALLBACK = False
EXTRA_ARGS: list[str] = []
PINNED_VERSIONS: dict[str, str] | None = None

for path in (DRIVE_ROOT, HF_HOME, LOG_DIR, ARTIFACT_ROOT, OUTPUT_DIR, Path(BASE_MODEL).parent):
    path.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME'] = str(HF_HOME)
os.environ['TRANSFORMERS_CACHE'] = str(HF_HOME / 'transformers')
os.environ['HUGGINGFACE_HUB_CACHE'] = str(HF_HOME / 'hub')
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['PYTHONHASHSEED'] = '48'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ.pop('HF_HUB_OFFLINE', None)
os.environ.pop('TRANSFORMERS_OFFLINE', None)

if TRAIN_PROFILE == 'current_colab':
    EXTRA_ARGS = [
        '--strict_determinism',
        '--disable_tf32',
        '--matmul_precision', 'highest',
    ]
elif TRAIN_PROFILE == 'parity_v28':
    EXTRA_ARGS = [
        '--strict_determinism',
        '--disable_tf32',
        '--matmul_precision', 'highest',
    ]
else:
    raise ValueError(f'Unsupported TRAIN_PROFILE: {TRAIN_PROFILE}')

{
    'REPO_URL': REPO_URL,
    'BRANCH': BRANCH,
    'WORKSPACE_DIR': str(WORKSPACE_DIR),
    'OUTPUT_DIR': str(OUTPUT_DIR),
    'RESUME_WARM_START_DIR': str(RESUME_WARM_START_DIR),
    'HF_BASE_MODEL_REPO': HF_BASE_MODEL_REPO,
    'HF_BASE_MODEL_REVISION': HF_BASE_MODEL_REVISION,
    'BASE_MODEL': BASE_MODEL,
    'TRAIN_PROFILE': TRAIN_PROFILE,
    'GPU_PROFILE': GPU_PROFILE,
    'RUN_BENCHMARK_AFTER_TRAIN': RUN_BENCHMARK_AFTER_TRAIN,
    'ENABLE_SQLITE_DATASET_MANIFEST': ENABLE_SQLITE_DATASET_MANIFEST,
    'ALLOW_BUNDLED_WARM_START_FALLBACK': ALLOW_BUNDLED_WARM_START_FALLBACK,
}


## Step 1 - Sync the repo and install dependencies

This notebook only needs the current training code, datasets, and warm-start artifacts. It keeps the workspace small with sparse checkout, but the actual trainer still runs from `source/qwen_supermix_pipeline.py`.


In [ ]:
import shutil
import subprocess
import sys

from importlib.metadata import PackageNotFoundError, version


def run(cmd: list[str], cwd: Path | None = None) -> None:
    print('+', ' '.join(cmd))
    try:
        subprocess.run(cmd, check=True, cwd=str(cwd) if cwd else None)
    except subprocess.CalledProcessError as exc:
        joined = ' '.join(cmd)
        raise RuntimeError(f'Command failed with exit code {exc.returncode}: {joined}') from exc


if WORKSPACE_DIR.exists():
    print(f'Removing existing workspace at {WORKSPACE_DIR} to avoid stale sparse-checkout state...')
    shutil.rmtree(WORKSPACE_DIR)

run([
    'git', 'clone', '--filter=blob:none', '--sparse', '--depth', '1', '--branch', BRANCH, REPO_URL, str(WORKSPACE_DIR)
])
run([
    'git', 'sparse-checkout', 'set',
    'source',
    'datasets',
    'artifacts/qwen_supermix_enhanced_v26_full',
    'dist/SupermixQwenDesktopV26/_internal/bundled_latest_artifact',
], cwd=WORKSPACE_DIR)

if shutil.which('git-lfs') is None:
    run(['apt-get', 'update'])
    run(['apt-get', 'install', '-y', 'git-lfs'])
run(['git', 'lfs', 'install'], cwd=WORKSPACE_DIR)
run([
    'git', 'lfs', 'pull',
    '--include=datasets/**,artifacts/qwen_supermix_enhanced_v26_full/**,dist/SupermixQwenDesktopV26/_internal/bundled_latest_artifact/**',
], cwd=WORKSPACE_DIR)

run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip', 'setuptools', 'wheel'])
run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'source/requirements_train_build.txt'], cwd=WORKSPACE_DIR)

extra_packages = [
    'transformers',
    'peft',
    'accelerate',
    'safetensors',
    'matplotlib',
    'tokenizers',
    'huggingface_hub',
    'sentencepiece',
    'tqdm',
]
if PINNED_VERSIONS:
    pinned = [f'{name}=={ver}' for name, ver in PINNED_VERSIONS.items()]
    run([sys.executable, '-m', 'pip', 'install', '-q', *pinned], cwd=WORKSPACE_DIR)
else:
    run([sys.executable, '-m', 'pip', 'install', '-q', *extra_packages], cwd=WORKSPACE_DIR)

from huggingface_hub import snapshot_download
import torch

resolved_versions = {}
for package_name in ['torch', 'transformers', 'peft', 'accelerate', 'safetensors', 'matplotlib', 'tokenizers', 'huggingface_hub']:
    try:
        resolved_versions[package_name] = version(package_name)
    except PackageNotFoundError:
        resolved_versions[package_name] = '<missing>'
print('Resolved package versions:', resolved_versions)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is not available after dependency installation.')
print('GPU:', torch.cuda.get_device_name(0))

snapshot_download(
    repo_id=HF_BASE_MODEL_REPO,
    revision=HF_BASE_MODEL_REVISION,
    local_dir=BASE_MODEL,
    local_dir_use_symlinks=False,
)
print('Workspace ready:', WORKSPACE_DIR)
print('Pinned base model path:', BASE_MODEL)


## Step 2 - Optional SQLite dataset manifest

Colab does not have a built-in training database for this Qwen pipeline, but Python includes `sqlite3`. This optional cell creates a Drive-backed dataset manifest for quick auditing, resumable file stats, and sanity checks without changing the training path.


In [ ]:
import hashlib
import sqlite3
from datetime import datetime, timezone

DATA_FILES = [
    'datasets/conversation_data.quality_anchor_v2.jsonl',
    'datasets/conversation_data.coding_knowledge_2026_02_19.jsonl',
    'datasets/conversation_data.world_events_2026_02_19.jsonl',
    'datasets/conversation_data.supermix_plus_v27_500k.jsonl',
    'datasets/conversation_data.mega_reasoning_creative_v25_75582.jsonl',
    'datasets/conversation_data.mega_creative_250k_v2.jsonl',
]

if ENABLE_SQLITE_DATASET_MANIFEST:
    SQLITE_MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(SQLITE_MANIFEST_PATH)
    conn.execute(
        '''
        CREATE TABLE IF NOT EXISTS dataset_manifest (
            rel_path TEXT PRIMARY KEY,
            row_count INTEGER NOT NULL,
            byte_count INTEGER NOT NULL,
            sha256 TEXT NOT NULL,
            updated_at_utc TEXT NOT NULL
        )
        '''
    )
    manifest_rows = []
    for rel_path in DATA_FILES:
        abs_path = WORKSPACE_DIR / rel_path
        if not abs_path.exists():
            raise FileNotFoundError(f'Missing dataset file: {abs_path}')
        sha256 = hashlib.sha256()
        with abs_path.open('rb') as fh:
            for chunk in iter(lambda: fh.read(1 << 20), b''):
                sha256.update(chunk)
        row_count = 0
        with abs_path.open('r', encoding='utf-8', errors='replace') as fh:
            for line in fh:
                if line.strip():
                    row_count += 1
        byte_count = abs_path.stat().st_size
        updated_at_utc = datetime.now(timezone.utc).isoformat()
        conn.execute(
            'REPLACE INTO dataset_manifest(rel_path, row_count, byte_count, sha256, updated_at_utc) VALUES (?, ?, ?, ?, ?)',
            (rel_path, row_count, byte_count, sha256.hexdigest(), updated_at_utc),
        )
        manifest_rows.append((rel_path, row_count, byte_count))
    conn.commit()
    print('SQLite manifest:', SQLITE_MANIFEST_PATH)
    for rel_path, row_count, byte_count in manifest_rows:
        print(f'- {rel_path}: rows={row_count} bytes={byte_count}')
    conn.close()
else:
    print('SQLite dataset manifest disabled.')


## Step 3 - Build the current training command

This cell keeps the same architecture because it still launches `source/qwen_supermix_pipeline.py`. The difference from the older Colab notebook is that it exposes current pipeline features such as grouped eval splitting, true SFT packing, newer distillation ranking controls, preference verbosity controls, and optional self-play knobs. It also auto-detects common Colab GPU tiers and only adjusts memory/throughput knobs, not the model family.


In [ ]:
import json
import os
import shlex
import shutil
import subprocess
import sys
from datetime import datetime, timezone

LOGICAL_CPU = max(1, os.cpu_count() or 1)
INTEROP_CPU = max(1, min(4, LOGICAL_CPU // 2))
REPO_WARM_START_DIR = WORKSPACE_DIR / 'artifacts' / 'qwen_supermix_enhanced_v26_full'
REPO_BUNDLED_WARM_START_DIR = WORKSPACE_DIR / 'dist' / 'SupermixQwenDesktopV26' / '_internal' / 'bundled_latest_artifact'


def get_latest_adapter_checkpoint(run_dir: Path) -> Path | None:
    if not run_dir.exists():
        return None
    direct_adapter = run_dir / 'adapter'
    if (direct_adapter / 'adapter_config.json').exists() and (direct_adapter / 'adapter_model.safetensors').exists():
        return direct_adapter.resolve()
    if (run_dir / 'adapter_config.json').exists() and (run_dir / 'adapter_model.safetensors').exists():
        return run_dir.resolve()
    latest_file = run_dir / 'latest_adapter_checkpoint.txt'
    if latest_file.exists():
        checkpoint_dir = latest_file.read_text(encoding='utf-8').strip()
        if checkpoint_dir:
            raw = Path(checkpoint_dir)
            candidates = [raw]
            if not raw.is_absolute():
                candidates.append(WORKSPACE_DIR / raw)
                candidates.append(run_dir / raw)
            for candidate in candidates:
                if candidate.exists():
                    return candidate.resolve()
    checkpoints_dir = run_dir / 'checkpoints'
    if not checkpoints_dir.exists():
        return None
    metas = sorted(checkpoints_dir.rglob('checkpoint_meta.json'), key=lambda path: path.stat().st_mtime, reverse=True)
    for meta in metas:
        adapter_dir = meta.parent / 'adapter'
        if adapter_dir.exists():
            return adapter_dir.resolve()
    return None


def seed_drive_warm_start() -> Path | None:
    drive_checkpoint = get_latest_adapter_checkpoint(RESUME_WARM_START_DIR)
    if drive_checkpoint is not None:
        return drive_checkpoint
    repo_checkpoint = get_latest_adapter_checkpoint(REPO_WARM_START_DIR)
    if repo_checkpoint is None:
        return None
    print('Seeding Drive warm-start artifact from repo checkout...')
    shutil.copytree(REPO_WARM_START_DIR, RESUME_WARM_START_DIR, dirs_exist_ok=True)
    return get_latest_adapter_checkpoint(RESUME_WARM_START_DIR)


def get_bundled_fallback_checkpoint() -> Path | None:
    return get_latest_adapter_checkpoint(REPO_BUNDLED_WARM_START_DIR)


def append_scalar_arg(cmd: list[str], flag: str, value: object) -> None:
    if value is None:
        return
    cmd.extend([flag, str(value)])


def append_bool_flag(cmd: list[str], flag: str, enabled: bool) -> None:
    if enabled:
        cmd.append(flag)


def detect_gpu_profile() -> tuple[str, str, int]:
    forced = str(GPU_PROFILE).strip().lower()
    if forced and forced != 'auto':
        return forced, forced.upper(), 0
    query = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'],
        text=True,
    ).splitlines()[0]
    gpu_name_raw, memory_mb_raw = [part.strip() for part in query.split(',', 1)]
    gpu_name = gpu_name_raw.upper()
    memory_mb = int(float(memory_mb_raw))
    if 'A100' in gpu_name:
        return 'a100', gpu_name_raw, memory_mb
    if 'L4' in gpu_name:
        return 'l4', gpu_name_raw, memory_mb
    if 'T4' in gpu_name:
        return 't4', gpu_name_raw, memory_mb
    return 'generic', gpu_name_raw, memory_mb


gpu_profile, detected_gpu_name, detected_gpu_memory_mb = detect_gpu_profile()
print(f'GPU profile: {gpu_profile} ({detected_gpu_name}, {detected_gpu_memory_mb} MiB)')


base_scalar_args = {
    '--max_records': 600000,
    '--max_source_fraction': 0.52,
    '--max_synthetic_fraction': 0.06,
    '--max_prompt_signature_count': 4,
    '--data_log_every_records': 2000,
    '--prompt_signature_cap_exempt_sources': 'conversation_data.quality_anchor_v2.jsonl,conversation_data.mega_reasoning_creative_v25_75582.jsonl',
    '--eval_size': 500,
    '--eval_split_mode': 'auto',
    '--eval_min_quality_score': 1.05,
    '--max_length': 448,
    '--batch_size': 1,
    '--grad_accum_steps': 16,
    '--epochs': 6,
    '--max_steps': 6200,
    '--lr': '1.0e-5',
    '--sft_lr_schedule': 'cosine_restarts',
    '--sft_lr_restart_period': 620,
    '--sft_warmup_steps': 30,
    '--sft_min_lr_ratio': 0.22,
    '--sft_max_grad_norm': 0.9,
    '--sft_focal_gamma': 1.35,
    '--sft_eval_every_steps': 240,
    '--sft_early_stop_patience': 5,
    '--sft_curriculum_quality_ramp': 0.22,
    '--sft_grad_noise_eta': 0.01,
    '--train_log_every_steps': 1,
    '--save_every_steps': SAVE_EVERY_STEPS,
    '--weight_decay': 0.02,
    '--lora_r': 32,
    '--lora_alpha': 64,
    '--lora_dropout': 0.03,
    '--lora_init': 'pissa_niter_4',
    '--lora_plus_ratio': 16,
    '--neftune_noise_alpha': 5.0,
    '--sft_weight_mode': 'quality',
    '--sft_min_weight': 0.62,
    '--sft_max_weight': 1.88,
    '--sft_synthetic_prompt_weight': 0.62,
    '--sft_teacher_source_weight': 0.92,
    '--sft_quality_anchor_boost': 1.14,
    '--sft_coding_boost': 1.24,
    '--sft_events_boost': 1.08,
    '--sft_reasoning_boost': 1.28,
    '--sft_prompt_skill_boost': 1.17,
    '--sft_conversation_boost': 1.24,
    '--sft_creativity_boost': 1.16,
    '--sft_knowledge_density_boost': 1.22,
    '--sft_rdrop_alpha': 0.05,
    '--sft_length_bucket_window_mult': 24,
    '--sft_followup_paraphrase_aug': 1,
    '--sft_followup_paraphrase_weight': 0.68,
    '--sft_min_quality_score': 0.98,
    '--sft_quality_filter_exempt_sources': 'conversation_data.quality_anchor_v2.jsonl,conversation_data.world_events_2026_02_19.jsonl',
    '--sft_source_balance_strength': 0.66,
    '--sft_source_balance_max_scale': 1.95,
    '--sft_packing_max_samples_per_row': 3,
    '--sft_selection_strategy': 'none',
    '--sft_selection_keep_ratio': 1.0,
    '--sft_selection_min_keep': 0,
    '--sft_selection_max_keep': 0,
    '--sft_selection_hardness_target': 0.45,
    '--sft_selection_hardness_bandwidth': 0.20,
    '--sft_selection_budget_mode': 'tokens',
    '--sft_selection_budget_power': 0.5,
    '--sft_selection_scope': 'all',
    '--sft_selection_scope_min_words': 8,
    '--preference_objective': 'ipo',
    '--preference_steps': 1500,
    '--preference_rescore_every': 25,
    '--preference_pairs': 34000,
    '--preference_candidate_count': 8,
    '--preference_reject_similarity_min': 0.16,
    '--preference_beta': 1.9,
    '--preference_beta_end': 3.6,
    '--preference_margin': 0.00,
    '--preference_margin_end': 0.00,
    '--preference_label_smoothing': 0.03,
    '--preference_sft_weight': 0.32,
    '--preference_length_weight': 0.08,
    '--preference_length_control_strength': 0.0,
    '--preference_length_control_target_ratio': 1.0,
    '--preference_length_control_max_penalty': 0.0,
    '--preference_hardness_gamma': 1.15,
    '--preference_robust_alpha': 0.30,
    '--preference_robust_eta': 0.08,
    '--preference_robust_clip': 2.5,
    '--preference_wpo_alpha': 0.35,
    '--preference_wpo_clip': 2.5,
    '--preference_reference_anchor_weight': 0.04,
    '--preference_reference_anchor_batch_size': 2,
    '--preference_short_reject_boost': 0.75,
    '--preference_long_reject_boost': 0.25,
    '--preference_min_chosen_quality': 0.92,
    '--preference_min_chosen_words': 8,
    '--preference_min_quality_gap': 0.05,
    '--preference_max_pairs_per_user': 2,
    '--preference_max_pairs_per_source': 360,
    '--preference_mining_mode': 'auto',
    '--preference_mining_progress_every': 30,
    '--preference_mining_max_seconds': 4500,
    '--preference_mining_max_attempt_factor': 20,
    '--preference_coding_focus_boost': 1.30,
    '--preference_reasoning_focus_boost': 1.32,
    '--preference_counterfactual_rejects_per_prompt': 4,
    '--preference_selection_strategy': 'innovation_mix',
    '--preference_selection_keep_ratio': 0.62,
    '--preference_selection_min_keep': 1800,
    '--preference_selection_max_keep': 2400,
    '--preference_selection_hardness_target': 0.46,
    '--preference_selection_hardness_bandwidth': 0.22,
    '--preference_length_bucket_window_mult': 24,
    '--preference_lr': '1.4e-5',
    '--preference_lr_schedule': 'cosine',
    '--preference_warmup_steps': 18,
    '--preference_min_lr_ratio': 0.30,
    '--preference_max_grad_norm': 0.9,
    '--preference_max_new_tokens': 112,
    '--preference_prompt_max_tokens': 352,
    '--preference_self_play_budget': 0,
    '--preference_self_play_curriculum': 'easy_to_hard',
    '--preference_self_play_max_new_tokens': 0,
    '--supermix_distill_ratio': 0.14,
    '--supermix_distill_max': 8000,
    '--supermix_distill_best_of': 3,
    '--supermix_distill_log_every': 40,
    '--supermix_distill_max_seconds': 12000,
    '--supermix_distill_min_quality': 0.93,
    '--supermix_distill_min_gain': 0.18,
    '--supermix_distill_density_bias': 0.20,
    '--supermix_distill_gain_bias': 0.0,
    '--supermix_distill_compactness_bias': 0.0,
    '--supermix_distill_rank_margin': 0.0,
    '--seed': 48,
    '--device_preference': 'cuda,npu,xpu,mps,cpu,dml',
    '--model_dtype': 'auto',
    '--torch_num_threads': LOGICAL_CPU,
    '--torch_interop_threads': INTEROP_CPU,
}

gpu_profile_scalar_overrides = {
    't4': {
        '--max_length': 384,
        '--grad_accum_steps': 24,
        '--preference_pairs': 24000,
        '--preference_candidate_count': 6,
        '--supermix_distill_max': 5000,
        '--sft_packing_max_samples_per_row': 2,
    },
    'l4': {
        '--max_length': 448,
        '--grad_accum_steps': 16,
        '--preference_pairs': 34000,
        '--preference_candidate_count': 8,
        '--supermix_distill_max': 8000,
        '--sft_packing_max_samples_per_row': 3,
    },
    'a100': {
        '--max_length': 512,
        '--grad_accum_steps': 12,
        '--preference_pairs': 42000,
        '--preference_candidate_count': 10,
        '--supermix_distill_max': 12000,
        '--sft_packing_max_samples_per_row': 4,
    },
    'generic': {},
}
profile_scalar_overrides = gpu_profile_scalar_overrides.get(gpu_profile, {})
if profile_scalar_overrides:
    print('Applying GPU-profile scalar overrides:', profile_scalar_overrides)
    base_scalar_args.update(profile_scalar_overrides)

base_bool_flags = {
    '--eval_drop_synthetic_prompts': True,
    '--use_rslora': True,
    '--use_dora': True,
    '--sft_length_bucketed_batches': True,
    '--sft_drop_synthetic_prompts': True,
    '--sft_auto_balance_sources': True,
    '--sft_true_packing': TRAIN_PROFILE == 'current_colab',
    '--preference_allow_template_prompts': True,
    '--preference_length_bucketed_batches': True,
    '--gradient_checkpointing': True,
}

optional_bool_flags = {
    '--supermix_distill_allow_synthetic_prompts': False,
}


def build_train_command() -> list[str]:
    cmd = [
        sys.executable,
        '-u',
        'source/qwen_supermix_pipeline.py',
        '--data',
        *DATA_FILES,
        '--base_model', BASE_MODEL,
        '--output_dir', str(OUTPUT_DIR),
        '--device', DEVICE,
    ]
    for flag, value in base_scalar_args.items():
        append_scalar_arg(cmd, flag, value)
    for flag, enabled in base_bool_flags.items():
        append_bool_flag(cmd, flag, enabled)
    for flag, enabled in optional_bool_flags.items():
        append_bool_flag(cmd, flag, enabled)

    if RUN_BENCHMARK_AFTER_TRAIN:
        append_scalar_arg(cmd, '--benchmark_eval_limit', BENCHMARK_EVAL_LIMIT)
    else:
        cmd.append('--skip_benchmark')

    latest_output_checkpoint = get_latest_adapter_checkpoint(OUTPUT_DIR)
    if latest_output_checkpoint is not None:
        print(f'Resuming from latest checkpoint in {OUTPUT_DIR}')
        cmd.append('--resume_from_latest_checkpoint')
    else:
        warm_start_checkpoint = seed_drive_warm_start()
        if warm_start_checkpoint is not None:
            print(f'Warm-starting from checkpoint in {RESUME_WARM_START_DIR}')
            cmd.extend(['--init_adapter_dir', str(warm_start_checkpoint), '--init_adapter_match_lora'])
        else:
            bundled_fallback = get_bundled_fallback_checkpoint() if ALLOW_BUNDLED_WARM_START_FALLBACK else None
            if bundled_fallback is not None:
                print(f'Using bundled warm-start fallback from {REPO_BUNDLED_WARM_START_DIR}')
                cmd.extend(['--init_adapter_dir', str(bundled_fallback), '--init_adapter_match_lora'])
            else:
                raise RuntimeError(
                    f'Warm-start adapter not found. Put qwen_supermix_enhanced_v26_full in {RESUME_WARM_START_DIR}, or set ALLOW_BUNDLED_WARM_START_FALLBACK = True.'
                )
    if EXTRA_ARGS:
        print('Applying extra args:', EXTRA_ARGS)
        cmd.extend(EXTRA_ARGS)
    return cmd


TRAIN_CMD = build_train_command()
OUT_LOG = LOG_DIR / f'train_{OUTPUT_DIR.name}.out.log'

launch_state = {
    'updated_at_utc': datetime.now(timezone.utc).isoformat(),
    'repo_url': REPO_URL,
    'branch': BRANCH,
    'workspace_dir': str(WORKSPACE_DIR),
    'output_dir': str(OUTPUT_DIR),
    'resume_warm_start_dir': str(RESUME_WARM_START_DIR),
    'base_model': BASE_MODEL,
    'device': DEVICE,
    'gpu_profile': gpu_profile,
    'detected_gpu_name': detected_gpu_name,
    'detected_gpu_memory_mb': detected_gpu_memory_mb,
    'out_log': str(OUT_LOG),
    'train_profile': TRAIN_PROFILE,
    'run_benchmark_after_train': RUN_BENCHMARK_AFTER_TRAIN,
    'command': TRAIN_CMD,
}
STATE_PATH.write_text(json.dumps(launch_state, indent=2), encoding='utf-8')
print('Command preview:')
print(' '.join(shlex.quote(part) for part in TRAIN_CMD))
print(json.dumps(launch_state, indent=2))


## Step 4 - Launch training

This cell streams training logs into the notebook and appends them to Drive. After a disconnect, rerun the config cell above and then rerun this one. The pipeline will resume from the latest checkpoint under `OUTPUT_DIR`.


In [ ]:
print('Streaming to:', OUT_LOG)
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

with OUT_LOG.open('a', encoding='utf-8') as log_file:
    process = subprocess.Popen(
        TRAIN_CMD,
        cwd=str(WORKSPACE_DIR),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        log_file.write(line)
        log_file.flush()
    return_code = process.wait()

if return_code != 0:
    raise RuntimeError(f'Training exited with code {return_code}')

print('Training finished successfully.')


## Step 5 - Reattach and inspect progress

This cell checks the latest checkpoint, the prepared-data cache, the teacher-distill cache, and the optional benchmark output so you can recover quickly after reconnects.


In [ ]:
from collections import deque
import json

latest_checkpoint = get_latest_adapter_checkpoint(OUTPUT_DIR)
print('Latest checkpoint:', latest_checkpoint)

if STATE_PATH.exists():
    launch_state = json.loads(STATE_PATH.read_text(encoding='utf-8'))
    print(
        'Launch runtime:',
        launch_state.get('train_profile'),
        launch_state.get('gpu_profile'),
        launch_state.get('detected_gpu_name'),
    )
else:
    print('No saved launch state yet.')

checkpoints_dir = OUTPUT_DIR / 'checkpoints'
if checkpoints_dir.exists():
    recent_checkpoints = sorted(checkpoints_dir.glob('*'), key=lambda path: path.stat().st_mtime, reverse=True)[:5]
    print('Recent checkpoint dirs:')
    for checkpoint_dir in recent_checkpoints:
        print(' -', checkpoint_dir)
else:
    print('No checkpoint directory yet.')

for cache_name in ['prepared_data_cache_meta.json', 'prepared_train_pairs.jsonl', 'prepared_eval_pairs.jsonl', 'teacher_distill_pairs.jsonl']:
    cache_path = OUTPUT_DIR / cache_name
    print(f'{cache_name}:', cache_path if cache_path.exists() else '<missing>')

if OUT_LOG.exists():
    with OUT_LOG.open('r', encoding='utf-8', errors='replace') as fh:
        tail = deque(fh, maxlen=80)
    print(''.join(tail))
else:
    print('No log file yet.')

benchmark_path = OUTPUT_DIR / 'benchmark_results.json'
if benchmark_path.exists():
    results = json.loads(benchmark_path.read_text(encoding='utf-8'))
    train_stats = results.get('train_stats', {})
    summary = {
        'eval_split_mode': train_stats.get('eval_split_mode'),
        'sft_true_packing': train_stats.get('sft_true_packing'),
        'sft_rows_before_packing': train_stats.get('sft_rows_before_packing'),
        'sft_rows_after_packing': train_stats.get('sft_rows_after_packing'),
        'sft_packed_rows': train_stats.get('sft_packed_rows'),
        'preference_objective': train_stats.get('preference_objective'),
        'teacher_generated': train_stats.get('teacher_generated'),
        'benchmark_sample_summary': results.get('sample_summary', {}),
    }
    print(json.dumps(summary, indent=2))
else:
    print('No benchmark_results.json yet. This is expected when RUN_BENCHMARK_AFTER_TRAIN = False or training has not finished.')
